# Часть 2. Обогащение данных из стороннего датасета

В этом ноутбуке мы загружаем очищенный датасет `netflix_clean_1.csv`.
И обогащаем его информацией о жанрах, бюджете, выручке, хронометраже, данные об аудитории и оценке из внешних источников.

Результат этой части - это обогащенный датасет `netflix_enriched_2.csv`, который используется в последующих частях анализа.

#### Выполнила Марокина Татьяна


## Шаг 1. Импорт библиотек и чтение датасетов


In [734]:
import pandas as pd
import numpy as np
import ast

### 1.1 Чтение "очищенного" датасета Netflix


In [735]:
df_cleared = pd.read_csv("netflix_clean_1.csv")
print(df_cleared.shape)
df_cleared.head(3)

(500, 6)


,title,rating,rating_level,release_year,rating_score,rating_size
0,White Chicks,PG-13,"crude and sexual humor, language and some drug...",2004,82.0,80
1,Lucky Number Slevin,R,"strong violence, sexual content and adult lang...",2006,NaN,82
2,Grey's Anatomy,TV-14,Parents strongly cautioned. May be unsuitable ...,2016,98.0,80


### 1.2 Чтение стороннего датасета для обогащения `TMDB_IMDB_Movies_Dataset.csv`

Датасет взят с [kaggle](https://www.kaggle.com/datasets/ggtejas/tmdb-imdb-merged-movies-dataset?select=TMDB++IMDB+Movies+Dataset.csv). Датаесет содержит 437246 строк и 29 признаков.


In [736]:
df_tmdb_imdb_movies = pd.read_csv("TMDB_IMDB_Movies_Dataset.csv")
print(df_tmdb_imdb_movies.shape)
df_tmdb_imdb_movies.head(3)

(437246, 29)


,id,title,vote_average,vote_count,status,release_date,revenue,runtime,adult,backdrop_path,...,genres,production_companies,production_countries,spoken_languages,keywords,directors,writers,averageRating,numVotes,cast
0,27205,Inception,8.364,34495,Released,2010-07-15,825532764,148,False,/8ZTVqvKDQ8emSGUEMjsS4yHAwrp.jpg,...,"Action, Science Fiction, Adventure","Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc...",Christopher Nolan,Christopher Nolan,8.8,2813028,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ken W..."
1,157336,Interstellar,8.417,32571,Released,2014-11-05,701729206,169,False,/pbrkL804c8yAv3zBZR4QPEafpAR.jpg,...,"Adventure, Drama, Science Fiction","Legendary Pictures, Syncopy, Lynda Obst Produc...","United Kingdom, United States of America",English,"rescue, future, spacecraft, race against time,...",Christopher Nolan,"Jonathan Nolan, Christopher Nolan",8.7,2523452,"Matthew McConaughey, Anne Hathaway, Michael Ca..."
2,155,The Dark Knight,8.512,30619,Released,2008-07-16,1004558444,152,False,/nMKdUUepR0i5zn0y1T4CsSB5chy.jpg,...,"Drama, Action, Crime, Thriller","DC Comics, Legendary Pictures, Syncopy, Isobel...","United Kingdom, United States of America","English, Mandarin","joker, sadism, chaos, secret identity, crime f...",Christopher Nolan,"Jonathan Nolan, Christopher Nolan, David S. Go...",9.1,3163410,"Christian Bale, Heath Ledger, Aaron Eckhart, M..."


### 1.3 Чтение стороннего датасета `movies_metadata.csv`

Датасет взят с [kaggle](https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset?select=movies_metadata.csv). Датаесет содержить 45466 строк и 24 признака.


In [737]:
df_mm = pd.read_csv("movies_metadata.csv", low_memory=False)
print(df_mm.shape)
df_mm.head(3)

(45466, 24)


,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0


## Шаг 2. Чистка датасетов и подготовка к слиянию


### 2.1 Подготовка `df_cleared`

Приготовим к слиянию датасет `df_cleared`.

- Убираем дубли по заголовку `title` и году выпуска `release_year` 
- Приводим год выпуска `release_year` к числу для дальнейшего сравнения
- Заголовок `title` приводим к нижнему регистру для точного сравнения
- Для признаков `rating_score` и `rating_size` длюавляем постфикс `_nfx` для различия


In [738]:
df_cleared["title_lower"] = df_cleared["title"].str.lower().str.strip()
df_cleared["release_year"] = df_cleared["release_year"].astype("Int64")

df_cleared = df_cleared.rename(
    columns={
        "rating_score": "rating_score_nfx",
        "rating_size": "rating_size_nfx",
    }
)

df_cleared = df_cleared.drop_duplicates(subset=["title", "release_year"]).reset_index(
    drop=True
)

print(df_cleared.shape)
df_cleared.head(3)

(499, 7)


,title,rating,rating_level,release_year,rating_score_nfx,rating_size_nfx,title_lower
0,White Chicks,PG-13,"crude and sexual humor, language and some drug...",2004,82.0,80,white chicks
1,Lucky Number Slevin,R,"strong violence, sexual content and adult lang...",2006,NaN,82,lucky number slevin
2,Grey's Anatomy,TV-14,Parents strongly cautioned. May be unsuitable ...,2016,98.0,80,grey's anatomy


### 2.2 Чистка `df_mm` и `df_tmdb_imdb_movies`

Приготовим к слиянию датасеты. Убираем лишние признаки, которые не планируем использовать в анализе


In [739]:
columns = [
    "title",
    "imdb_id",
    "budget",
    "revenue",
    "runtime",
    "genres",
    "vote_average",
    "vote_count",
    "release_date",
    'popularity',
]
df_mm = df_mm[columns].copy()

columns = [
    "title",
    "tconst",
    "budget",
    "revenue",
    "runtime",
    "genres",
    "vote_average",
    "vote_count",
    "release_date",
    "averageRating",
    "numVotes",
    'popularity',
]
df_tmdb_imdb_movies = df_tmdb_imdb_movies[columns].copy()

Для избежания ошибок в слиянии датасетов необходимо избавиться полностью от строк без названия. Все остальные признаки можно пока оставить как есть, поскольку избавляться от пропусков, если они есть, мы должны будем после слияния. Пока-что мы просто пытаемся обогатить исходный датасет информацией.

- Избавимся также от всех пропусков по полю `title` и дублей по полю `title` + `release_year`. Для целей обогащения такие строки попросту не нужны.
- Дополнительно подготовим столбец с годом `release_year`, для дальнейшего слияния в паре с названием `title`.
- переименуем столбцы для единообразия и разделения источников (для рейтинга)


In [740]:
df_mm = df_mm.dropna(subset=["title"]).copy()
df_mm["title"] = df_mm["title"].str.lower()

df_mm = df_mm.rename(columns={"vote_average": "vote_average_tmdb"})
df_mm = df_mm.rename(columns={"vote_count": "vote_count_tmdb"})

df_mm["release_year"] = pd.to_datetime(df_mm["release_date"], errors="coerce").dt.year
df_mm["release_year"] = df_mm["release_year"].astype("Int64")
df_mm = (
    df_mm.drop(columns=["release_date"])
    .drop_duplicates(subset=["title", "release_year"])
    .reset_index(drop=True)
)

print(df_mm.shape)
df_mm.head(3)

(45379, 10)


,title,imdb_id,budget,revenue,runtime,genres,vote_average_tmdb,vote_count_tmdb,popularity,release_year
0,toy story,tt0114709,30000000,373554033.0,81.0,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",7.7,5415.0,21.946943,1995
1,jumanji,tt0113497,65000000,262797249.0,104.0,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",6.9,2413.0,17.015539,1995
2,grumpier old men,tt0113228,0,0.0,101.0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",6.5,92.0,11.7129,1995


In [741]:
df_tmdb_imdb_movies = df_tmdb_imdb_movies.dropna(subset=["title"]).copy()
df_tmdb_imdb_movies["title"] = df_tmdb_imdb_movies["title"].str.lower()

df_tmdb_imdb_movies = df_tmdb_imdb_movies.rename(columns={"tconst": "imdb_id"})
df_tmdb_imdb_movies = df_tmdb_imdb_movies.rename(
    columns={"averageRating": "vote_average_imdb"}
)
df_tmdb_imdb_movies = df_tmdb_imdb_movies.rename(
    columns={"numVotes": "vote_count_imdb"}
)
df_tmdb_imdb_movies = df_tmdb_imdb_movies.rename(
    columns={"vote_average": "vote_average_tmdb"}
)
df_tmdb_imdb_movies = df_tmdb_imdb_movies.rename(
    columns={"vote_count": "vote_count_tmdb"}
)

df_tmdb_imdb_movies["release_year"] = pd.to_datetime(
    df_tmdb_imdb_movies["release_date"], errors="coerce"
).dt.year
df_tmdb_imdb_movies["release_year"] = df_tmdb_imdb_movies["release_year"].astype(
    "Int64"
)
df_tmdb_imdb_movies = (
    df_tmdb_imdb_movies.drop(columns=["release_date"])
    .drop_duplicates(subset=["title", "release_year"])
    .reset_index(drop=True)
)

print(df_tmdb_imdb_movies.shape)
df_tmdb_imdb_movies.head(3)

(424395, 12)


,title,imdb_id,budget,revenue,runtime,genres,vote_average_tmdb,vote_count_tmdb,vote_average_imdb,vote_count_imdb,popularity,release_year
0,inception,tt1375666,160000000,825532764,148,"Action, Science Fiction, Adventure",8.364,34495,8.8,2813028,83.952,2010
1,interstellar,tt0816692,165000000,701729206,169,"Adventure, Drama, Science Fiction",8.417,32571,8.7,2523452,140.241,2014
2,the dark knight,tt0468569,185000000,1004558444,152,"Drama, Action, Crime, Thriller",8.512,30619,9.1,3163410,130.643,2008


Также можно сразу заметить одну странность: в иных строках `budget` и `revenue` отмечено 0. Это явный пропуск, поскольку производство фильма не может стоить 0 и принести 0 дохода


In [742]:
df_mm[df_mm["revenue"] == 0].head(3)

,title,imdb_id,budget,revenue,runtime,genres,vote_average_tmdb,vote_count_tmdb,popularity,release_year
2,grumpier old men,tt0113228,0,0.0,101.0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",6.5,92.0,11.7129,1995
6,sabrina,tt0114319,58000000,0.0,127.0,"[{'id': 35, 'name': 'Comedy'}, {'id': 10749, '...",6.2,141.0,6.677277,1995
7,tom and huck,tt0112302,0,0.0,97.0,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...",5.4,45.0,2.561161,1995


Избавимся от пропусков по полям `budget` и `revenue`, явно указав `NAN`


In [743]:
df_mm["budget"] = df_mm["budget"].apply(lambda x: np.nan if int(x) == 0 else int(x))
df_mm["revenue"] = df_mm["revenue"].apply(lambda x: np.nan if int(x) == 0 else int(x))

df_tmdb_imdb_movies["budget"] = df_tmdb_imdb_movies["budget"].apply(
    lambda x: np.nan if int(x) == 0 else int(x)
)
df_tmdb_imdb_movies["revenue"] = df_tmdb_imdb_movies["revenue"].apply(
    lambda x: np.nan if int(x) == 0 else int(x)
)

In [744]:
df_mm.isna().sum()

title                    0
imdb_id                 17
budget               36506
revenue              37981
runtime                256
genres                   0
vote_average_tmdb        0
vote_count_tmdb          0
popularity               0
release_year            84
dtype: int64

In [745]:
df_tmdb_imdb_movies.isna().sum()

title                     0
imdb_id                   0
budget               395854
revenue              408259
runtime                   0
genres                74669
vote_average_tmdb         0
vote_count_tmdb           0
vote_average_imdb         0
vote_count_imdb           0
popularity                0
release_year          17274
dtype: int64

По некоторым признакам пропусков слишком много. В данный момент ничего с ними не будем делать и проведем слияние датасетов, и потом оценить результат слияния


Преобразуем поле `genres` в датасете `df_mm`. Стоит отметить, что оно записано не в виде JSON, а в форме простого python literal.

ПРиведем его к виду - строка с жанрами через запятую и пробел, как это записанов датасете `df_tmdb_imdb_movies`


In [746]:
def extract_name(x: str, field: str) -> str | float:
    if pd.isna(x) or x == "":
        return np.nan
    try:
        genres_json = ast.literal_eval(x)
        result = []

        for genre in genres_json:
            if isinstance(genre, dict) and field in genre:
                result.append(genre[field])

        return ", ".join(result) if result else np.nan

    except:
        return np.nan


df_mm["genres"] = df_mm["genres"].apply(lambda x: extract_name(str(x), "name"))

In [747]:
df_mm.isna().sum()

title                    0
imdb_id                 17
budget               36506
revenue              37981
runtime                256
genres                2440
vote_average_tmdb        0
vote_count_tmdb          0
popularity               0
release_year            84
dtype: int64

In [748]:
df_mm.head(3)

,title,imdb_id,budget,revenue,runtime,genres,vote_average_tmdb,vote_count_tmdb,popularity,release_year
0,toy story,tt0114709,30000000.0,373554033.0,81.0,"Animation, Comedy, Family",7.7,5415.0,21.946943,1995
1,jumanji,tt0113497,65000000.0,262797249.0,104.0,"Adventure, Fantasy, Family",6.9,2413.0,17.015539,1995
2,grumpier old men,tt0113228,NaN,NaN,101.0,"Romance, Comedy",6.5,92.0,11.7129,1995


## Шаг 3. Слияние датасетов


Будем поочередно мерджить исходный датасет `df_cleared` сначала с датасетом `df_tmdb_imdb_movies`, затем с датасетом `df_mm`, чтобы максимально получить новых данных и сократить количество возможных пропусков.


### 3.1 Мерджим `df_cleared` с `df_tmdb_imdb_movies` по `title` + `release_year`

Слияние будем выполнять по заголовку `title` и году выпуска контента `release_year`


In [749]:
df_cleared_copy = df_cleared.copy()

df_merged = pd.merge(
    df_cleared_copy,
    df_tmdb_imdb_movies,
    left_on=["title_lower", "release_year"],
    right_on=["title", "release_year"],
    how="left",
    indicator=True,
)
print(df_merged.shape)
df_merged.head(3)

(499, 19)


,title_x,rating,rating_level,release_year,rating_score_nfx,rating_size_nfx,title_lower,title_y,imdb_id,budget,revenue,runtime,genres,vote_average_tmdb,vote_count_tmdb,vote_average_imdb,vote_count_imdb,popularity,_merge
0,White Chicks,PG-13,"crude and sexual humor, language and some drug...",2004,82.0,80,white chicks,white chicks,tt0381707,37000000.0,113086475.0,109.0,"Comedy, Crime",6.919,3505.0,5.9,193002.0,54.851,both
1,Lucky Number Slevin,R,"strong violence, sexual content and adult lang...",2006,NaN,82,lucky number slevin,lucky number slevin,tt0425210,27000000.0,56308881.0,110.0,"Drama, Thriller, Crime, Mystery",7.479,3740.0,7.7,338824.0,21.607,both
2,Grey's Anatomy,TV-14,Parents strongly cautioned. May be unsuitable ...,2016,98.0,80,grey's anatomy,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only


Убираем лишние столбцы после мерджа и переименовываем оставшиеся. И проверяем сколько нашлось совпадений `matched_count`


In [750]:
df_merged = df_merged.rename(
    columns={"title_x": "title", "release_year_x": "release_year"}
)
df_merged["_merge_tmdb"] = df_merged["_merge"]
df_merged = df_merged.drop(columns=["_merge", "title_y"])

matched_count = (df_merged["_merge_tmdb"] == "both").sum()
print(f"3.1: смерджено {matched_count} записей по title + year")
df_merged.head(3)

3.1: смерджено 190 записей по title + year


,title,rating,rating_level,release_year,rating_score_nfx,rating_size_nfx,title_lower,imdb_id,budget,revenue,runtime,genres,vote_average_tmdb,vote_count_tmdb,vote_average_imdb,vote_count_imdb,popularity,_merge_tmdb
0,White Chicks,PG-13,"crude and sexual humor, language and some drug...",2004,82.0,80,white chicks,tt0381707,37000000.0,113086475.0,109.0,"Comedy, Crime",6.919,3505.0,5.9,193002.0,54.851,both
1,Lucky Number Slevin,R,"strong violence, sexual content and adult lang...",2006,NaN,82,lucky number slevin,tt0425210,27000000.0,56308881.0,110.0,"Drama, Thriller, Crime, Mystery",7.479,3740.0,7.7,338824.0,21.607,both
2,Grey's Anatomy,TV-14,Parents strongly cautioned. May be unsuitable ...,2016,98.0,80,grey's anatomy,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only


Проверим, что одинаковые названия но разные года выпуска смержены корректно


In [751]:
duplicated_titles_list = df_merged["title_lower"][
    df_merged.duplicated(subset=["title_lower"], keep=False)
].unique()
print(duplicated_titles_list)

<StringArray>
['skins', 'star wars: the clone wars', 'goosebumps']
Length: 3, dtype: str


In [752]:
df_tmdb_imdb_movies[df_tmdb_imdb_movies["title"] == "skins"]

,title,imdb_id,budget,revenue,runtime,genres,vote_average_tmdb,vote_count_tmdb,vote_average_imdb,vote_count_imdb,popularity,release_year
6612,skins,tt5808778,NaN,NaN,78,"Drama, Comedy",6.455,479,6.0,7975,13.346,2017
77180,skins,tt0284494,NaN,249204.0,84,"Action, Drama",4.700,10,7.0,1679,3.880,2002
312711,skins,tt0120143,NaN,NaN,14,"Comedy, Science Fiction",0.000,0,7.2,22,0.740,1997


In [753]:
df_merged[df_merged["title_lower"] == "skins"]

,title,rating,rating_level,release_year,rating_score_nfx,rating_size_nfx,title_lower,imdb_id,budget,revenue,runtime,genres,vote_average_tmdb,vote_count_tmdb,vote_average_imdb,vote_count_imdb,popularity,_merge_tmdb
119,Skins,TV-MA,For mature audiences. May not be suitable for...,2013,NaN,82,skins,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
135,Skins,TV-MA,For mature audiences. May not be suitable for...,2017,NaN,82,skins,tt5808778,NaN,NaN,78.0,"Drama, Comedy",6.455,479.0,6.0,7975.0,13.346,both


In [754]:
df_merged.isna().sum()

title                  0
rating                 0
rating_level           0
release_year           0
rating_score_nfx     243
rating_size_nfx        0
title_lower            0
imdb_id              309
budget               401
revenue              401
runtime              309
genres               311
vote_average_tmdb    309
vote_count_tmdb      309
vote_average_imdb    309
vote_count_imdb      309
popularity           309
_merge_tmdb            0
dtype: int64

### 3.2 Мерджим незаполненные в `df_merged` с `df_tmdb_imdb_movies` по `title` + `release_year` в диапазоне +- один год

В ходе работы с датасетами обнаружилось, что встречаются одинаковые фильмы по названию, но год не совпадает с разницей в +- один год. Сложно ответить, с чем это связано.

Поэтому слияние будем выполнять также по заголовку `title` и по году выпуска контента `release_year` +- один год

Выполним это только для строк, которые не были смерджены ранее `_merge_tmdb` == `left_only`


In [755]:
df_merged[df_merged["title_lower"] == "lottie dottie chicken"]

,title,rating,rating_level,release_year,rating_score_nfx,rating_size_nfx,title_lower,imdb_id,budget,revenue,runtime,genres,vote_average_tmdb,vote_count_tmdb,vote_average_imdb,vote_count_imdb,popularity,_merge_tmdb
15,Lottie Dottie Chicken,TV-Y,Suitable for all ages.,2009,NaN,82,lottie dottie chicken,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only


In [756]:
df_tmdb_imdb_movies[df_tmdb_imdb_movies["title"] == "lottie dottie chicken"]

,title,imdb_id,budget,revenue,runtime,genres,vote_average_tmdb,vote_count_tmdb,vote_average_imdb,vote_count_imdb,popularity,release_year
88309,lottie dottie chicken,tt2299730,NaN,NaN,30,"Animation, Music",9.8,8,6.3,66,1.715,2008


In [757]:
df_with_tmdb = df_merged[df_merged["_merge_tmdb"] == "both"].copy()
df_without_tmdb = df_merged[df_merged["_merge_tmdb"] == "left_only"].copy()

df_without_tmdb_copy = df_without_tmdb.drop(
    columns=[
        "revenue",
        "runtime",
        "_merge_tmdb",
        "imdb_id",
        "budget",
        "genres",
        "vote_average_tmdb",
        "vote_count_tmdb",
        "vote_average_imdb",
        "vote_count_imdb",
        'popularity',
    ]
).copy()
print(df_without_tmdb_copy.shape)
df_without_tmdb_copy.head(3)

(309, 7)


,title,rating,rating_level,release_year,rating_score_nfx,rating_size_nfx,title_lower
2,Grey's Anatomy,TV-14,Parents strongly cautioned. May be unsuitable ...,2016,98.0,80,grey's anatomy
3,Prison Break,TV-14,Parents strongly cautioned. May be unsuitable ...,2008,98.0,80,prison break
4,How I Met Your Mother,TV-PG,Parental guidance suggested. May not be suitab...,2014,94.0,80,how i met your mother


In [758]:
df_unmatched_merged = pd.merge(
    df_without_tmdb_copy,
    df_tmdb_imdb_movies,
    left_on="title_lower",
    right_on="title",
    how="left",
    indicator=True,
    suffixes=("_netflix", "_tmdb"),
)

# Проверяем диапазон года (+-1 год от release_year)
year_in_range = df_unmatched_merged["release_year_tmdb"].isna() | (
    (
        df_unmatched_merged["release_year_tmdb"]
        >= df_unmatched_merged["release_year_netflix"] - 1
    )
    & (
        df_unmatched_merged["release_year_tmdb"]
        <= df_unmatched_merged["release_year_netflix"] + 1
    )
)

# То, что смерджилось но не в диапазоне, заполняем NaN
df_unmatched_merged.loc[
    ~year_in_range,
    [
        "revenue",
        "runtime",
        "imdb_id",
        "budget",
        "genres",
        "vote_average_tmdb",
        "vote_count_tmdb",
        "vote_average_imdb",
        "vote_count_imdb",
        "popularity"
    ],
] = np.nan


df_unmatched_merged = df_unmatched_merged.drop(
    columns=["title_tmdb", "release_year_tmdb"]
)
df_unmatched_merged = df_unmatched_merged.rename(
    columns={"title_netflix": "title", "release_year_netflix": "release_year"}
)

df_unmatched_merged["_merge_tmdb"] = df_unmatched_merged["_merge"]
df_unmatched_merged = df_unmatched_merged.drop(columns=["_merge"])

matched_count = (
    (df_unmatched_merged["_merge_tmdb"] == "both")
    & (df_unmatched_merged["vote_average_tmdb"].notna())
).sum()
print(
    f"3.2: смерджено еще {matched_count} записей по title с годом в диапазоне +-1 год"
)

df_unmatched_merged = df_unmatched_merged.drop_duplicates(
    subset=["title", "release_year"]
).reset_index(drop=True)
print(df_unmatched_merged.shape)

3.2: смерджено еще 36 записей по title с годом в диапазоне +-1 год
(309, 18)


In [759]:
df_merged = pd.concat([df_with_tmdb, df_unmatched_merged], ignore_index=True)
df_merged = df_merged.drop(columns=["_merge_tmdb"])
print(df_merged.shape)
df_merged.isna().sum()

(499, 17)


title                  0
rating                 0
rating_level           0
release_year           0
rating_score_nfx     243
rating_size_nfx        0
title_lower            0
imdb_id              283
budget               391
revenue              395
runtime              283
genres               291
vote_average_tmdb    283
vote_count_tmdb      283
vote_average_imdb    283
vote_count_imdb      283
popularity           283
dtype: int64

In [760]:
df_merged.head(5)

,title,rating,rating_level,release_year,rating_score_nfx,rating_size_nfx,title_lower,imdb_id,budget,revenue,runtime,genres,vote_average_tmdb,vote_count_tmdb,vote_average_imdb,vote_count_imdb,popularity
0,White Chicks,PG-13,"crude and sexual humor, language and some drug...",2004,82.0,80,white chicks,tt0381707,37000000.0,113086475.0,109.0,"Comedy, Crime",6.919,3505.0,5.9,193002.0,54.851
1,Lucky Number Slevin,R,"strong violence, sexual content and adult lang...",2006,NaN,82,lucky number slevin,tt0425210,27000000.0,56308881.0,110.0,"Drama, Thriller, Crime, Mystery",7.479,3740.0,7.7,338824.0,21.607
2,Death Note,TV-14,Parents strongly cautioned. May be unsuitable ...,2006,77.0,80,death note,tt0758742,20000000.0,29667169.0,126.0,"Fantasy, Mystery, Thriller",6.949,614.0,7.5,33333.0,17.425
3,The Hunter,R,language and brief violence,2011,NaN,82,the hunter,tt1703148,NaN,176669.0,102.0,"Drama, Thriller, Adventure",6.620,594.0,6.7,42712.0,11.980
4,The Do-Over,TV-MA,For mature audiences. May not be suitable for...,2016,84.0,80,the do-over,tt4769836,NaN,NaN,108.0,"Action, Adventure, Comedy",5.697,1399.0,5.7,57047.0,11.916


### 3.3 Мерджим незаполненные в `df_merged` с `df_mm` по `title` + `release_year` в диапазоне +- один год

Пропуски остались. Возможно, второй датасет позволит заполнить пробелы.

Проведем слияние со вторым датасетом.


In [761]:
df_with_data = df_merged[df_merged["vote_average_tmdb"].notna()].copy()
df_without_data = df_merged[df_merged["vote_average_tmdb"].isna()].copy()
df_without_data = df_without_data.drop(
    columns=[
        "vote_average_tmdb",
        "vote_count_tmdb",
        "revenue",
        "runtime",
        "imdb_id",
        "budget",
        "genres",
        "popularity",
    ]
)
df_without_data.shape

(283, 9)

In [762]:
df_unmatched_merged = pd.merge(
    df_without_data,
    df_mm,
    left_on="title_lower",
    right_on="title",
    how="left",
    indicator=True,
    suffixes=("_netflix", "_mm"),
)

# Проверяем диапазон года (+-1 год от release_year)
year_in_range = df_unmatched_merged["release_year_mm"].isna() | (
    (
        df_unmatched_merged["release_year_mm"]
        >= df_unmatched_merged["release_year_netflix"] - 1
    )
    & (
        df_unmatched_merged["release_year_mm"]
        <= df_unmatched_merged["release_year_netflix"] + 1
    )
)

# То, что смерджилось но не в диапазоне, заполняем NaN
df_unmatched_merged.loc[
    ~year_in_range,
    [
        "vote_average_tmdb",
        "vote_count_tmdb",
        "revenue",
        "runtime",
        "imdb_id",
        "budget",
        "genres",
        "popularity",
    ],
] = np.nan


df_unmatched_merged = df_unmatched_merged.drop(columns=["title_mm", "release_year_mm"])
df_unmatched_merged = df_unmatched_merged.rename(
    columns={"title_netflix": "title", "release_year_netflix": "release_year"}
)

df_unmatched_merged["_merge_mm"] = df_unmatched_merged["_merge"]
df_unmatched_merged = df_unmatched_merged.drop(columns=["_merge"])

matched_count = (
    (df_unmatched_merged["_merge_mm"] == "both")
    & (df_unmatched_merged["vote_average_tmdb"].notna())
).sum()
print(
    f"3.3: смерджено еще {matched_count} записей по title с годом в диапазоне +-1 год"
)

df_unmatched_merged = df_unmatched_merged.drop_duplicates(
    subset=["title", "release_year"]
).reset_index(drop=True)
print(df_unmatched_merged.shape)

3.3: смерджено еще 2 записей по title с годом в диапазоне +-1 год
(283, 18)


In [763]:
df_merged = pd.concat([df_with_data, df_unmatched_merged], ignore_index=True)
df_merged = df_merged.drop(columns=["_merge_mm"])
print(df_merged.shape)

(499, 17)


In [764]:
df_merged.head(3)

,title,rating,rating_level,release_year,rating_score_nfx,rating_size_nfx,title_lower,imdb_id,budget,revenue,runtime,genres,vote_average_tmdb,vote_count_tmdb,vote_average_imdb,vote_count_imdb,popularity
0,White Chicks,PG-13,"crude and sexual humor, language and some drug...",2004,82.0,80,white chicks,tt0381707,37000000.0,113086475.0,109.0,"Comedy, Crime",6.919,3505.0,5.9,193002.0,54.851
1,Lucky Number Slevin,R,"strong violence, sexual content and adult lang...",2006,NaN,82,lucky number slevin,tt0425210,27000000.0,56308881.0,110.0,"Drama, Thriller, Crime, Mystery",7.479,3740.0,7.7,338824.0,21.607
2,Death Note,TV-14,Parents strongly cautioned. May be unsuitable ...,2006,77.0,80,death note,tt0758742,20000000.0,29667169.0,126.0,"Fantasy, Mystery, Thriller",6.949,614.0,7.5,33333.0,17.425


## Шаг 4. Заполнение пропусков в финальном датасете


Проверим, сколько осталось пропусков в результате


In [765]:
df_merged.isna().sum()

title                  0
rating                 0
rating_level           0
release_year           0
rating_score_nfx     243
rating_size_nfx        0
title_lower            0
imdb_id              281
budget               390
revenue              395
runtime              281
genres               289
vote_average_tmdb    281
vote_count_tmdb      281
vote_average_imdb    283
vote_count_imdb      283
popularity           281
dtype: int64

Таким образом, в некоторых полях по-прежнему имеются пропуски.


### 4.1 Заполнение пропусков в `genres`, `runtime` и `imdb_id`

Подключим дополнительны [датасет](https://developer.imdb.com/non-commercial-datasets/?utm_source=chatgpt.com#titlebasicstsvgz) с официального сайта IMDB, который содержит информацию о жанрах, времени контента


In [766]:
df_imdb = pd.read_csv("title.basics.tsv", sep="\t", na_values=["\\N"], low_memory=False)

In [767]:
df_imdb.head(3)

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
0,tt0000001,short,Carmencita,Carmencita,0,1894.0,NaN,1,"Documentary,Short"
1,tt0000002,short,Le clown et ses chiens,Le clown et ses chiens,0,1892.0,NaN,5,"Animation,Short"
2,tt0000003,short,Poor Pierrot,Pauvre Pierrot,0,1892.0,NaN,5,"Animation,Comedy,Romance"


Подготовим данные к слиянию

- Определим нужные поля
- Подготовим заголовок для слияния (приведем к нижнему регистру)
- Колонку с жанрами отформатируем как в итоговом датасете


In [768]:
df_imdb_copy = df_imdb[
    ["tconst", "primaryTitle", "startYear", "endYear", "runtimeMinutes", "genres"]
].copy()

In [769]:
df_imdb_copy["runtimeMinutes"] = pd.to_numeric(
    df_imdb_copy["runtimeMinutes"], errors="coerce"
).astype("Int64")

df_imdb_copy["title_lower"] = df_imdb_copy["primaryTitle"].str.lower().str.strip()
df_imdb_copy = df_imdb_copy.drop(columns=["primaryTitle"])

df_imdb_copy["genres"] = df_imdb_copy["genres"].str.replace(",", ", ")

print(df_imdb_copy.shape)
df_imdb_copy.head(5)

(12475319, 6)


,tconst,startYear,endYear,runtimeMinutes,genres,title_lower
0,tt0000001,1894.0,NaN,1,"Documentary, Short",carmencita
1,tt0000002,1892.0,NaN,5,"Animation, Short",le clown et ses chiens
2,tt0000003,1892.0,NaN,5,"Animation, Comedy, Romance",poor pierrot
3,tt0000004,1892.0,NaN,12,"Animation, Short",un bon bock
4,tt0000005,1893.0,NaN,1,Short,blacksmith scene


Выполним слияние по заголовку


In [770]:
df_merged_imdb = pd.merge(
    df_merged,
    df_imdb_copy,
    left_on="title_lower",
    right_on="title_lower",
    how="left",
    suffixes=("_netflix", "_imdb"),
)

Определим функцию, с помощью которой будем проверять входит ли год release_year в исходном датасете в диапазон годов из датасета IMDB, так как слияние только по заголовкам может быть не точным.

`startYear` или `endYear` должны быть близки к `release_year`. Диапазон как и ранее +- один год.


In [771]:
def check_year_match(row):
    if pd.isna(row["release_year"]):
        return True

    netflix_year = int(row["release_year"])
    start_year = row["startYear"]
    end_year = row["endYear"]

    if pd.notna(start_year):
        if abs(int(start_year) - netflix_year) <= 1:
            return True

    if pd.notna(end_year):
        if abs(int(end_year) - netflix_year) <= 1:
            return True

    return False

Применить проверку года. И выведем итоговый датасет.


In [772]:
df_merged_imdb["year_match"] = df_merged_imdb.apply(check_year_match, axis=1)

# Обнулим данные если год не подходит
cols_to_nullify = ["tconst", "runtimeMinutes", "genres_imdb"]
for col in cols_to_nullify:
    if col in df_merged_imdb.columns:
        df_merged_imdb.loc[~df_merged_imdb["year_match"], col] = np.nan

# Заполним пропуски
df_merged_imdb["imdb_id"] = df_merged_imdb["imdb_id"].fillna(df_merged_imdb["tconst"])
df_merged_imdb["runtime"] = df_merged_imdb["runtime"].fillna(
    df_merged_imdb["runtimeMinutes"]
)
df_merged_imdb["genres"] = df_merged_imdb["genres_netflix"].fillna(
    df_merged_imdb["genres_imdb"]
)

df_final = df_merged_imdb.drop(
    columns=[
        "startYear",
        "endYear",
        "tconst",
        "runtimeMinutes",
        "genres_imdb",
        "year_match",
        "genres_netflix",
    ]
)

df_final = df_final.drop_duplicates(subset=["title", "release_year"]).reset_index(
    drop=True
)

print(df_final.shape)

(499, 17)


In [773]:
df_final.isna().sum()

title                  0
rating                 0
rating_level           0
release_year           0
rating_score_nfx     243
rating_size_nfx        0
title_lower            0
imdb_id              175
budget               390
revenue              395
runtime              184
vote_average_tmdb    281
vote_count_tmdb      281
vote_average_imdb    283
vote_count_imdb      283
popularity           281
genres               179
dtype: int64

Благодаря датасету IMDB удалось существенно сократить количество пропусков в полях `genres`, `runtime`, `imdb_id`


Заполним пропуски по полю `genres`.
Для этого применим метод Мультиметки по консолидированным жанрам. Такой вариант пригодится нам в последующем анализе.

- Выделим подгруппы жанров,
- добавим соответствующие бинарные столбцы.


In [774]:
def parse_all_genres(genres_str):
    if pd.isna(genres_str) or genres_str == "":
        return []

    return [g.strip() for g in str(genres_str).split(", ")]


genre_mapper = {
    "Action": "Action",
    "Crime": "Action",
    "Adventure": "Action",
    "Thriller": "Action",
    "Western": "Action",
    "Comedy": "Comedy",
    "Drama": "Drama",
    "Romance": "Drama",
    "Family": "Family",
    "Animation": "Family",
    "Children": "Family",
    "Documentary": "Family",
    "Horror": "Other",
    "Fantasy": "Other",
    "Sci-Fi": "Other",
    "Science Fiction": "Other",
    "Mystery": "Other",
    "Music": "Other",
}

df_final["genres_list"] = df_final["genres"].apply(parse_all_genres)
df_final["genres_consolidated"] = df_final["genres_list"].apply(
    lambda genres_list: list(
        set([genre_mapper.get(g, "Other") for g in genres_list if g])
    )
)

for genre in ["Action", "Comedy", "Drama", "Family", "Other"]:
    df_final[f"is_{genre.lower()}"] = df_final["genres_consolidated"].apply(
        lambda x: 1 if genre in x else 0
    )

Итого распределено по жанрам


In [775]:
for genre in ["Action", "Comedy", "Drama", "Family", "Other"]:
    count = df_final[f"is_{genre.lower()}"].sum()
    print(f"{genre}: {count}")

Action: 135
Comedy: 166
Drama: 113
Family: 188
Other: 83


Мультиметка реализована с помощью бинарных колонок (one-hot encoding). Каждый фильм может иметь несколько жанров из набора `{Action, Comedy, Drama, Family, Other}`.


In [776]:
df_final.head(3)

,title,rating,rating_level,release_year,rating_score_nfx,rating_size_nfx,title_lower,imdb_id,budget,revenue,...,vote_count_imdb,popularity,genres,genres_list,genres_consolidated,is_action,is_comedy,is_drama,is_family,is_other
0,White Chicks,PG-13,"crude and sexual humor, language and some drug...",2004,82.0,80,white chicks,tt0381707,37000000.0,113086475.0,...,193002.0,54.851,"Comedy, Crime","[Comedy, Crime]","[Action, Comedy]",1,1,0,0,0
1,Lucky Number Slevin,R,"strong violence, sexual content and adult lang...",2006,NaN,82,lucky number slevin,tt0425210,27000000.0,56308881.0,...,338824.0,21.607,"Drama, Thriller, Crime, Mystery","[Drama, Thriller, Crime, Mystery]","[Action, Other, Drama]",1,0,1,0,1
2,Death Note,TV-14,Parents strongly cautioned. May be unsuitable ...,2006,77.0,80,death note,tt0758742,20000000.0,29667169.0,...,33333.0,17.425,"Fantasy, Mystery, Thriller","[Fantasy, Mystery, Thriller]","[Action, Other]",1,0,0,0,1


Было решено не заполнять пропуски в столбцах `budget` и `revenue`. Во-первых, они составляют 60 % от всего датасета. Во-вторых, не существует определенной зависимости между ними и другими признаками, или недостаточно данных для ее выявления. Бюджет зависит прежде всего от кинопроизводителя и актерского состава — 500 строк недостаточно, чтобы обучить эффективную модель.


In [777]:
df_final.corr(numeric_only=True)

,release_year,rating_score_nfx,rating_size_nfx,budget,revenue,runtime,vote_average_tmdb,vote_count_tmdb,vote_average_imdb,vote_count_imdb,is_action,is_comedy,is_drama,is_family,is_other
release_year,1.000000,0.200235,0.056086,0.130263,0.207611,-0.123955,-0.179297,-0.116036,-0.145504,-0.065973,-0.172004,-0.172469,-0.052097,-0.385020,-0.129398
rating_score_nfx,0.200235,1.000000,NaN,0.551200,0.510393,-0.104636,-0.086563,0.393985,0.085387,0.283561,0.005097,-0.135044,-0.131401,-0.082182,0.066797
rating_size_nfx,0.056086,NaN,1.000000,-0.376524,-0.344272,-0.043514,-0.128633,-0.491526,-0.302443,-0.385036,-0.044959,-0.000043,-0.050895,0.143658,-0.067463
budget,0.130263,0.551200,-0.376524,1.000000,0.674964,0.037076,0.181835,0.589417,0.229409,0.456247,0.331803,0.036362,-0.261333,0.329952,0.142483
revenue,0.207611,0.510393,-0.344272,0.674964,1.000000,-0.009306,0.136208,0.763585,0.154685,0.607244,0.246833,0.223303,-0.217251,0.199301,-0.063319
runtime,-0.123955,-0.104636,-0.043514,0.037076,-0.009306,1.000000,-0.315294,-0.004162,0.164587,-0.015123,0.012167,0.136437,0.103036,-0.031583,0.082987
vote_average_tmdb,-0.179297,-0.086563,-0.128633,0.181835,0.136208,-0.315294,1.000000,0.302801,0.277320,-0.043057,0.234771,-0.046891,0.079955,0.198259,0.100440
vote_count_tmdb,-0.116036,0.393985,-0.491526,0.589417,0.763585,-0.004162,0.302801,1.000000,0.314173,0.620747,0.261790,0.058520,-0.018163,0.139934,0.179017
vote_average_imdb,-0.145504,0.085387,-0.302443,0.229409,0.154685,0.164587,0.277320,0.314173,1.000000,0.394974,0.193594,-0.224342,0.211985,0.016174,0.113298
vote_count_imdb,-0.065973,0.283561,-0.385036,0.456247,0.607244,-0.015123,-0.043057,0.620747,0.394974,1.000000,0.094781,-0.020446,0.093149,-0.045445,0.168904


In [778]:
df_final.isna().sum()

title                    0
rating                   0
rating_level             0
release_year             0
rating_score_nfx       243
rating_size_nfx          0
title_lower              0
imdb_id                175
budget                 390
revenue                395
runtime                184
vote_average_tmdb      281
vote_count_tmdb        281
vote_average_imdb      283
vote_count_imdb        283
popularity             281
genres                 179
genres_list              0
genres_consolidated      0
is_action                0
is_comedy                0
is_drama                 0
is_family                0
is_other                 0
dtype: int64

In [779]:
df_final.to_csv("netflix_enriched_2.csv", index=False)